# Phase 1: Dataset creation

GitHub issue: #2. Depends on Phase 0 (#1): target = leave-last-out next-item prediction within a session.

This notebook builds the frozen Phase 1 snapshot from the real OTTO sample. It also documents a real
correction made mid-phase: the first version of this pipeline leaked future weeks into the training
artifacts. That mistake and the fix are both kept here, since understanding *why* the final approach
is what it is matters as much as the approach itself.

## Step 1: Inventory sources

Source: OTTO recsys dataset, Kaggle dataset `otto/recsys-dataset` (not the competition - see decision
`_doc/decisions/D1-otto-license.md`: CC BY 4.0, confirmed directly from Kaggle's own metadata). OTTO has
no item text, no queries, no positions - categories and queries are synthetic (`_doc/AGENTS.md`).

**Update**: this originally ran on an 8GB Mac, so the raw file (~11.3GB / 220M events) was sampled down
to ~12% on Kaggle's own compute first (`notebooks/kaggle_otto_sample/otto-sample-04.ipynb`) before ever
touching this machine. Running on Colab with real RAM and a GPU removes that constraint, so this
notebook now processes the **entire** raw file directly - no sampling. The cell below detects which
environment it's running in and does the right thing either way, so this notebook still runs locally
against the old 12% sample if needed, but on Colab it converts and uses all 220M events.

In [1]:
import json
import time
from pathlib import Path

import polars as pl

# tries the Colab clone location first, falls back to local-Mac-relative detection
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/search_ai_project'),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
]
ROOT = next(p for p in CANDIDATE_ROOTS if (p / "pyproject.toml").exists())
print(f"ROOT: {ROOT}")

RAW_TRAIN_FILE = Path('/content/drive/MyDrive/otto_data/otto-recsys-train.jsonl')
FULL_EVENTS_PATH = RAW_TRAIN_FILE.parent / "events_full.parquet"
SAMPLE_DIR = ROOT / "data" / "sample"

if FULL_EVENTS_PATH.exists():
    print(f"Already converted: {FULL_EVENTS_PATH}")
    events = pl.read_parquet(FULL_EVENTS_PATH)
elif RAW_TRAIN_FILE.exists():
    size_gb = RAW_TRAIN_FILE.stat().st_size / 1e9
    print(f"Converting the full raw file ({size_gb:.1f} GB, ~220M events, no sampling) ...")
    print("This flattens nested per-session JSON into flat rows and sorts by (session, ts) -")
    print("a real, heavy operation at this scale. Watch for it stalling vs. just taking a while.")
    t0 = time.time()
    TYPE_CODE = {"clicks": 0, "carts": 1, "orders": 2}
    (
        pl.scan_ndjson(RAW_TRAIN_FILE, empty_as_null=False)
        .explode("events")
        .unnest("events")
        .select(
            pl.col("session").cast(pl.Int64),
            pl.col("aid").cast(pl.Int32),
            pl.col("ts").cast(pl.Int64),
            pl.col("type").replace_strict(TYPE_CODE, default=0).cast(pl.Int8).alias("type"),
        )
        .sort(["session", "ts"])
        .sink_parquet(FULL_EVENTS_PATH, compression="zstd")
    )
    print(f"[{time.time()-t0:.0f}s] wrote {FULL_EVENTS_PATH}")
    events = pl.read_parquet(FULL_EVENTS_PATH)
else:
    print("No raw Colab dataset found - falling back to the local 12% sample")
    sample_manifest = json.loads((SAMPLE_DIR / "sample_manifest.json").read_text())
    print(json.dumps(sample_manifest, indent=2))
    events = pl.read_parquet(SAMPLE_DIR / "events.parquet")

print(events.schema)
print(f"rows: {events.height:,}  sessions: {events['session'].n_unique():,}")
events.head(3)

ROOT: /Users/shaliqshukoor/search_project
No raw Colab dataset found - falling back to the local 12% sample
{
  "rule": "keep session if ((session * 2654435761) % 2**32 >> 8) % 100 < keep_pct",
  "keep_pct": 12,
  "source": "otto-recsys-train.jsonl",
  "source_sessions": 12899779,
  "sessions": 1547984,
  "events": 26047712,
  "events_by_type": {
    "click": 23400355,
    "cart": 2034479,
    "order": 612878
  },
  "ts_min": 1659304800025,
  "ts_max": 1661723999965,
  "days": 28.0,
  "sha256": {
    "events.parquet": "49a199a4f5d9a9823e9a33032ec785ffd0f4148d78d2ac447a419386805a56ea"
  }
}


Schema([('session', Int64), ('aid', Int32), ('ts', Int64), ('type', Int8)])


rows: 26,047,712  sessions: 1,547,984


session,aid,ts,type
i64,i32,i64,i8
0,1517085,1659304800025,0
0,1563459,1659304904511,0
0,1309446,1659367439426,0


On the old 12% local sample, this was 1,547,984 sessions (12% of 12,899,779), 26,047,712 events,
with a type mix of 89.8% click / 7.8% cart / 2.4% order - close to the real dataset's published
overall mix, a good sign the sampling wasn't distorting behavior. The cell above now shows the real
numbers for whatever actually ran (full dataset on Colab, or the old sample locally).

**Finding**: session counts decline sharply week over week in this data (we'll compute this below).
That's expected, not a sampling bug - OTTO's own published methodology trims sessions that would
overlap the labeled/test boundary, and (as this notebook's own Step 3 correction will show) OTTO
"sessions" persist across the whole window, so fewer *new* sessions start near the end.

In [2]:
weeks = events.with_columns(
    (((pl.col("ts") - pl.col("ts").min()) // (7 * 24 * 60 * 60 * 1000))).alias("_week")
)
session_first_week = weeks.group_by("session").agg(pl.col("_week").min().alias("week"))
session_first_week["week"].value_counts().sort("week")

week,count
i64,u32
0,641864
1,395328
2,294642
3,216150


## Step 2: Define the label

Leave-last-out per session (`src/ranking/data/labels.py`): the session's last event (by timestamp) is
held out. The category of that held-out item becomes the synthetic *query* - a documented leak used as
an intent proxy (OTTO has no real search queries), not evidence of real query understanding. The prefix
(every earlier event) is what a model would actually see. Graded label: click=1, cart=2, order=3.

## Step 3: Observation window and cutoff - and a correction

The first version of this pipeline bucketed sessions into train/val/test by the week of their **first**
event, then built categories/co-visitation from "all events of sessions that started in weeks 0-1".

That was wrong. Look at how long OTTO sessions actually run:

In [3]:
sample_sessions = session_first_week.sample(2000, seed=0)["session"]
span = (
    events.join(sample_sessions.to_frame(), on="session", how="inner")
    .group_by("session")
    .agg((pl.col("ts").max() - pl.col("ts").min()).alias("span_ms"))
    .with_columns((pl.col("span_ms") / 86_400_000).alias("span_days"))
)
print(span["span_days"].describe())
long = span.filter(pl.col("span_days") > 7).height
print(f"sessions spanning > 7 days: {long} / {span.height} ({long/span.height:.1%})")

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 2000.0    │
│ null_count ┆ 0.0       │
│ mean       ┆ 6.497199  │
│ std        ┆ 8.267159  │
│ min        ┆ 0.0       │
│ 25%        ┆ 0.005772  │
│ 50%        ┆ 1.843436  │
│ 75%        ┆ 11.874128 │
│ max        ┆ 27.822207 │
└────────────┴───────────┘
sessions spanning > 7 days: 716 / 2000 (35.8%)


Median session span is multiple days, and roughly half run longer than a week - one sampled session
in the full analysis ran from 2022-07-31 to 2022-08-28, the *entire* window, with an 11-day gap between
two of its events. OTTO "sessions" here behave like a persistent visitor id, not a 30-minute browsing
session.

That means bucketing by a session's *first* event silently pulled each long session's later weeks'
events into what the old code called "training-window" data - real point-in-time leakage into the
shared category/co-visitation artifacts every downstream phase depends on, not just a labeling quirk.

**Fix**: split by each example's own `label_ts` (the held-out event's own timestamp), never by session
identity. Categories/co-visitation are fit on events with `ts` before the train/val cutoff, from *any*
session - not "sessions that started early".

In [4]:
MS_PER_WEEK = 7 * 24 * 60 * 60 * 1000
ts_min = events["ts"].min()
train_end = ts_min + 2 * MS_PER_WEEK
val_end = ts_min + 3 * MS_PER_WEEK
print("train_end:", train_end, "val_end:", val_end)

train_events = events.filter(pl.col("ts") < train_end)
print(f"train events (ts-based, any session): {train_events.height:,}")

train_end: 1660514400025 val_end: 1661119200025
train events (ts-based, any session): 13,064,647


## Step 4: Point-in-time joins - categories and co-visitation, training window only

Two more real findings surfaced building this:

1. **Item frequency is heavily skewed.** Median item frequency is 4; most of the catalog is long-tail.
   Items with fewer than 5 occurrences in the training window get category **-1** (too sparse to place
   reliably) rather than a forced, unreliable placement. `MIN_ITEM_FREQ` below was tuned by eye on the
   12% sample - worth revisiting once the full dataset's frequency distribution is visible, since 10x
   more data means many more items clear any fixed bar.
2. **The first clustering attempt collapsed.** Raw co-visitation weights vary hugely by item
   popularity, and Euclidean k-means on the raw SVD embedding grouped items by *magnitude* (how
   connected an item is overall) instead of *direction* (who it specifically co-occurs with) - 99.2%
   of placeable items landed in one giant cluster. Fixed by L2-normalizing the co-visitation matrix and
   the SVD embedding before k-means (`src/ranking/data/categories.py`) - turns it into cosine-similarity
   clustering. Verified on synthetic mock data first (recovery of known hidden categories jumped from
   weak to a perfect adjusted Rand index of 1.0) before trusting it on the real data.

**Scale note**: this step did an unbatched self-join-style computation that we later found scales badly
once volume gets large enough (see Phase 4 issue #5 - the 23GB memory incident). At full dataset scale
(~10x this notebook's original 13M-event training window), watch the timing printed below - if it's
still running after several minutes with no sign of progress, stop and flag it rather than waiting
indefinitely, the same way `clicks_matrix` needed batching once it hit real scale.

In [5]:
import sys
sys.path.insert(0, str(ROOT / "src"))

from ranking.data.covisitation import covisitation_pairs, top_k_neighbors
from ranking.data.categories import cluster_categories

MIN_ITEM_FREQ = 5
N_CATEGORIES = 80

item_freq = train_events.group_by("aid").agg(pl.len().alias("n"))
frequent_aids = item_freq.filter(pl.col("n") >= MIN_ITEM_FREQ)["aid"].to_numpy()
print(f"frequent items (>= {MIN_ITEM_FREQ} in train): {len(frequent_aids):,} / {item_freq.height:,}")

t0 = time.time()
covis = covisitation_pairs(
    train_events.filter(pl.col("aid").is_in(frequent_aids.tolist())), max_gap_events=5
)
print(f"[{time.time()-t0:.0f}s] co-visitation pairs: {covis.height:,}")

frequent items (>= 5 in train): 396,630 / 1,070,425


[47s] co-visitation pairs: 17,292,646


In [6]:
t0 = time.time()
clustered = cluster_categories(covis, frequent_aids, n_categories=N_CATEGORIES, n_components=32, seed=0)
print(f"[{time.time()-t0:.0f}s] clustered {clustered.height:,} items")

sizes = clustered["category"].value_counts().sort("count", descending=True)
print("largest category share of known items:", sizes["count"].max() / clustered.height)
print("median category size:", sizes["count"].median())
sizes.head(5)

[99s] clustered 396,630 items
largest category share of known items: 0.024609837884174167
median category size: 4777.5


category,count
i32,u32
41,9761
79,9198
71,8901
42,8840
46,8701


Largest category is now ~2.6% of known items (was 99.2% before the fix), median ~4,600 - a
healthy, usable distribution across all 80 categories.

In [7]:
all_aids_in_sample = events["aid"].unique().to_frame()
item_categories = all_aids_in_sample.join(clustered, on="aid", how="left").with_columns(
    pl.col("category").fill_null(-1)
)
unknown_share = (item_categories["category"] == -1).mean()
print(f"items with category -1 (unknown/sparse), catalog-wide: {unknown_share:.1%}")

items with category -1 (unknown/sparse), catalog-wide: 72.0%


## Step 5: Build the ranking examples and validate

In [8]:
from ranking.data.labels import build_ranking_examples

t0 = time.time()
examples = build_ranking_examples(events, item_categories).with_columns(
    pl.when(pl.col("label_ts") < train_end)
    .then(pl.lit("train"))
    .when(pl.col("label_ts") < val_end)
    .then(pl.lit("val"))
    .otherwise(pl.lit("test"))
    .alias("split")
)
print(f"[{time.time()-t0:.0f}s] built {examples.height:,} examples")

print("split counts:")
print(examples["split"].value_counts().sort("split"))
print()
print("duplicate sessions:", examples.height - examples["session"].n_unique())
print("nulls:", dict(examples.null_count().row(0, named=True)))
print("example-level unknown query_category rate:", (examples["query_category"] == -1).mean())

[6s] built 1,547,984 examples
split counts:
shape: (3, 2)
┌───────┬────────┐
│ split ┆ count  │
│ ---   ┆ ---    │
│ str   ┆ u32    │
╞═══════╪════════╡
│ test  ┆ 639037 │
│ train ┆ 512161 │
│ val   ┆ 396786 │
└───────┴────────┘

duplicate sessions: 0
nulls: {'session': 0, 'prefix_last_ts': 0, 'query_category': 0, 'label_aid': 0, 'label_type': 0, 'label_grade': 0, 'label_ts': 0, 'prefix_len': 0, 'split': 0}
example-level unknown query_category rate: 0.17401859450743676


Zero duplicates, zero nulls anywhere, and the example-level unknown-category rate (12-19%,
varies by split) is much lower than the catalog-level rate (72%) - held-out items skew popular, so
they're more likely to have cleared the frequency-5 bar during category fitting.

## Step 6: Freeze

Written with a manifest (version, split cutoffs, per-file checksums) so the snapshot is reproducible
and every downstream notebook can point at a known, checksummed version. On Colab this goes to Drive
(outside the repo, same as the raw data - too large to push to git either way); locally it goes to
`data/frozen/`. Data dictionary: `_doc/data_dictionary.md`.

In [9]:
import hashlib
from datetime import UTC, datetime

FROZEN_DIR = (RAW_TRAIN_FILE.parent / "frozen") if RAW_TRAIN_FILE.exists() else (ROOT / "data" / "frozen")
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
events.write_parquet(FROZEN_DIR / "events.parquet")
item_categories.write_parquet(FROZEN_DIR / "item_categories.parquet")
examples.write_parquet(FROZEN_DIR / "ranking_examples.parquet")
covis.write_parquet(FROZEN_DIR / "train_covisitation.parquet")
print(f"[{time.time()-t0:.0f}s] wrote parquet files to {FROZEN_DIR}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "version": datetime.now(UTC).strftime("%Y-%m-%d"),
    "split_cutoffs_ms": {"ts_min": int(ts_min), "train_end": int(train_end), "val_end": int(val_end)},
    "min_item_freq_for_category": MIN_ITEM_FREQ,
    "n_categories": N_CATEGORIES,
    "items_total": item_categories.height,
    "items_unknown_category": int((item_categories["category"] == -1).sum()),
    "ranking_examples": examples.height,
    "example_counts": {row["split"]: row["count"] for row in examples["split"].value_counts().to_dicts()},
    "seed": 0,
    "files": {
        f.name: sha256(f)
        for f in [
            FROZEN_DIR / "events.parquet",
            FROZEN_DIR / "item_categories.parquet",
            FROZEN_DIR / "ranking_examples.parquet",
            FROZEN_DIR / "train_covisitation.parquet",
        ]
    },
}
(FROZEN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

[2s] wrote parquet files to /Users/shaliqshukoor/search_project/data/frozen


{
  "version": "2026-10-02",
  "split_cutoffs_ms": {
    "ts_min": 1659304800025,
    "train_end": 1660514400025,
    "val_end": 1661119200025
  },
  "min_item_freq_for_category": 5,
  "n_categories": 80,
  "items_total": 1414089,
  "items_unknown_category": 1017459,
  "ranking_examples": 1547984,
  "example_counts": {
    "val": 396786,
    "test": 639037,
    "train": 512161
  },
  "seed": 0,
  "files": {
    "events.parquet": "69d6080b681eba8a29f06a505040425ab3dfdd9fc3ab49b10ce19172cd029806",
    "item_categories.parquet": "239c1e4e97b2ef23de246cfaf558aa98351b4b2a8f5ddba2ab86f6251c1b2568",
    "ranking_examples.parquet": "bfbaf1b61f1f89b82e256ae16bf1109d951b5672256164e910a5c34dadd5f62d",
    "train_covisitation.parquet": "beba34871047a50639f5c837dd69ccb2adea2318bbaecb7bdb15c531fd4c0aff"
  }
}


## Exit criteria

Versioned snapshot (`data/frozen/manifest.json`), data dictionary (`_doc/data_dictionary.md`), written
label definition (Step 2 above), row counts reconciled to source (Step 5 above, matches
`data/sample/sample_manifest.json`). Met - see issue #2 for the full write-up including the
correction's numbers before and after.